# Turn Sentinel-1 images into training-format tiles

Takes GeoTIFFs downloaded and cuts them into tiles in **exactly the same format as the xView3 training tiles**, so a model trained on `xview3_tiles` sees familiar data:

* 640 × 640 PNG, R = VV, G = VH, B = mean(R, G)
* each band: `uint8 = 1 + 254 × clip((dB − lo) / (hi − lo), 0, 1)`; 0 means *no data*
* `lo` / `hi`: VV −30 … +10 dB, VH −35 … +5 dB (fixed in this notebook, the same values the training
  tiles were made with)
* tiles overlap by `TILE_OVERLAP` px (default 64) so a ship on a tile edge is whole in at least one tile

For every pass folder it writes:

```
<pass folder>/tiles/<area>_<YYYYMMDDHHMM>_<row>_<col>.png
<pass folder>/tiles.csv     tile id, file, pixel offset, UTM corner, centre lat/lon, time, coverage
<pass folder>/tiles.json    the scaling and settings used
```

The last section shows how to turn a detection's pixel position into latitude/longitude.

## Settings
`None` means "use `.env`".

In [ ]:
ENV_FILE = None       # path to .env; None = look in this folder (and its parent folders)
FOLDERS = None        # e.g. ["data/sentinel1/rotterdam/2026-10-28_0544"]; None = every pass in S1_DATA_DIR
OVERLAP = None       # tile overlap in px; None = TILE_OVERLAP from .env (default 64)
FORCE = False         # True = re-tile passes that already have tiles (e.g. after changing the scaling)

## Encoding of the training tiles
These must be identical to the values the `xview3_tiles` training set was made with (the defaults of
`xview3_prep.py`, also listed in `xview3_tiles/meta/summary.json`). Only change them if you rebuilt
the training set with other values, otherwise the model sees unfamiliar data.

In [ ]:
VV_DB_RANGE = (-30.0, 10.0)   # VV dB mapped to pixel values 1..255
VH_DB_RANGE = (-35.0, 5.0)    # VH dB mapped to pixel values 1..255
TILE_SIZE = 640               # px

## Imports and `.env`

In [ ]:
import csv
import json
import math
import os
import shutil

import numpy as np
import tifffile
from PIL import Image

DEFAULT_DATA_DIR = os.path.join("data", "sentinel1")


class StopRun(Exception):
    """A problem that needs your attention."""


def find_env(explicit=None):
    if explicit:
        return explicit
    folder = os.getcwd()
    for _ in range(3):
        p = os.path.join(folder, ".env")
        if os.path.isfile(p):
            return p
        folder = os.path.dirname(folder)
    return None


def load_env(path):
    values = {}
    if not path or not os.path.isfile(path):
        return values
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            if line.startswith("export "):
                line = line[7:]
            key, val = line.split("=", 1)
            val = val.strip()
            if len(val) >= 2 and val[0] == val[-1] and val[0] in "\"'":
                val = val[1:-1]
            elif " #" in val:
                val = val.split(" #", 1)[0].strip()
            values[key.strip()] = val
    return values

## Coordinates
`tile_pixel_to_latlon` converts a pixel inside a tile to latitude/longitude using that tile's row in
`tiles.csv`. Copy this cell into your detection notebook to place detections on the map.

In [ ]:
_A, _F, _K0 = 6378137.0, 1 / 298.257223563, 0.9996
_N = _F / (2 - _F)
_AA = _A / (1 + _N) * (1 + _N ** 2 / 4 + _N ** 4 / 64)
_BETA = (_N / 2 - 2 * _N ** 2 / 3 + 37 * _N ** 3 / 96, _N ** 2 / 48 + _N ** 3 / 15, 17 * _N ** 3 / 480)
_DELTA = (2 * _N - 2 * _N ** 2 / 3 - 2 * _N ** 3, 7 * _N ** 2 / 3 - 8 * _N ** 3 / 5, 56 * _N ** 3 / 15)


def utm_to_latlon(easting, northing, zone=31):
    xi = northing / (_K0 * _AA)
    eta = (easting - 500000) / (_K0 * _AA)
    xi_ = xi - sum(b * math.sin(2 * j * xi) * math.cosh(2 * j * eta) for j, b in enumerate(_BETA, 1))
    eta_ = eta - sum(b * math.cos(2 * j * xi) * math.sinh(2 * j * eta) for j, b in enumerate(_BETA, 1))
    chi = math.asin(math.sin(xi_) / math.cosh(eta_))
    phi = chi + sum(d * math.sin(2 * j * chi) for j, d in enumerate(_DELTA, 1))
    lam = math.atan2(math.sinh(eta_), math.cos(xi_))
    return math.degrees(phi), zone * 6 - 183 + math.degrees(lam)


def tile_pixel_to_latlon(tile, x, y):
    """tile: a row of tiles.csv (dict); x, y: pixel position inside the tile -> (lat, lon)."""
    px = float(tile.get("pixel_m", 10))
    e = float(tile["utm_x0"]) + (x + 0.5) * px
    n = float(tile["utm_y0"]) - (y + 0.5) * px
    return utm_to_latlon(e, n, int(tile.get("utm_zone", 31)))

## Reading the GeoTIFFs
The map position comes from `scene.json` (written by notebook 1), or from the GeoTIFF's own map tags
for files that came from elsewhere.

In [ ]:
def read_georef(folder, vv_path):
    """(utm_zone, x0, y0, pixel_m, acquired_utc, area)"""
    meta_path = os.path.join(folder, "scene.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            m = json.load(f)
        epsg = int(str(m["crs"]).split(":")[-1])
        x0, y0 = m["utm_upper_left"]
        return epsg - 32600, float(x0), float(y0), float(m.get("pixel_m", 10)), m.get("acquired_utc", ""), m.get("area")
    with tifffile.TiffFile(vv_path) as tf:
        g = tf.geotiff_metadata or {}
    scale, tie = g.get("ModelPixelScale"), g.get("ModelTiepoint")
    epsg = g.get("ProjectedCSTypeGeoKey")
    epsg = int(getattr(epsg, "value", epsg) or 0)
    if not (scale and tie and 32601 <= epsg <= 32660):
        raise ValueError(f"{vv_path}: no scene.json and no UTM-north GeoTIFF georeference")
    return (epsg - 32600, float(tie[3]) - float(tie[0]) * scale[0], float(tie[4]) + float(tie[1]) * scale[1],
            float(scale[0]), "", None)


def read_band(path):
    """Float32 array in dB with no-data as NaN."""
    a = tifffile.imread(path).astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0] if a.shape[-1] < a.shape[0] else a[0]
    with tifffile.TiffFile(path) as tf:
        tag = tf.pages[0].tags.get(42113)
    nodata = None
    if tag is not None:
        try:
            nodata = float(str(tag.value).strip("\x00 "))
        except ValueError:
            pass
    bad = ~np.isfinite(a) | (a <= -1000)
    if nodata is not None:
        bad |= a == nodata
    a[bad] = np.nan
    return a

## Encoding (identical to the xView3 tile preparation)

In [ ]:
def scale(db, lo, hi):
    v = np.clip((db - lo) / (hi - lo), 0, 1) * 254 + 1
    return np.where(np.isfinite(db), v, 0).astype(np.uint8)


def to_rgb(vv, vh, vv_rng, vh_rng):
    r, g = scale(vv, *vv_rng), scale(vh, *vh_rng)
    b = ((r.astype(np.uint16) + g) // 2).astype(np.uint8)
    b[(r == 0) | (g == 0)] = 0
    return np.dstack([r, g, b])

## Tiling

In [ ]:
def tile_folder(folder, vv_rng, vh_rng, T, overlap):
    vv_path, vh_path = os.path.join(folder, "VV_dB.tif"), os.path.join(folder, "VH_dB.tif")
    zone, x0, y0, px, acquired, area = read_georef(folder, vv_path)
    vv, vh = read_band(vv_path), read_band(vh_path)
    if vv.shape != vh.shape:
        raise ValueError("VV and VH have different sizes")
    H, W = vv.shape
    rgb = to_rgb(vv, vh, vv_rng, vh_rng)

    stamp = acquired.replace("-", "").replace(":", "").replace("T", "")[:12]
    prefix = f"{area}_{stamp}" if area else os.path.basename(os.path.abspath(folder))
    tiles_dir = os.path.join(folder, "tiles")
    if os.path.isdir(tiles_dir):
        shutil.rmtree(tiles_dir)
    os.makedirs(tiles_dir)

    stride = T - overlap
    rows = []
    for r in range(0, max(H - overlap, 1), stride):
        for c in range(0, max(W - overlap, 1), stride):
            tile = np.zeros((T, T, 3), np.uint8)
            part = rgb[r:r + T, c:c + T]
            tile[:part.shape[0], :part.shape[1]] = part
            vfrac = float((tile[..., 0] > 0).mean())
            if vfrac < 0.01:
                continue
            tid = f"{prefix}_{r // stride:03d}_{c // stride:03d}"
            Image.fromarray(tile).save(os.path.join(tiles_dir, tid + ".png"))
            ux, uy = x0 + c * px, y0 - r * px
            clat, clon = utm_to_latlon(ux + T * px / 2, uy - T * px / 2, zone)
            rows.append({"tile_id": tid, "file": f"tiles/{tid}.png", "pixel_row": r, "pixel_col": c,
                         "utm_zone": zone, "utm_x0": ux, "utm_y0": uy, "pixel_m": px,
                         "center_lat": round(clat, 6), "center_lon": round(clon, 6),
                         "valid_fraction": round(vfrac, 3), "acquired_utc": acquired})
    if not rows:
        raise ValueError("no tiles with data")
    with open(os.path.join(folder, "tiles.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    with open(os.path.join(folder, "tiles.json"), "w") as f:
        json.dump({"tile_size": T, "overlap": overlap,
                   "channels": {"R": "VV dB", "G": "VH dB", "B": "mean of R and G"},
                   "scaling": {"VV_dB_range": list(vv_rng), "VH_dB_range": list(vh_rng),
                               "formula": "uint8 = 1 + 254 * clip((dB - lo) / (hi - lo), 0, 1); 0 = no data"},
                   "tiles": len(rows),
                   "pixel_to_latlon": "E = utm_x0 + (x + 0.5) * pixel_m; N = utm_y0 - (y + 0.5) * pixel_m; "
                                      "then UTM (utm_zone, north) -> lat/lon"}, f, indent=2)
    return len(rows), (W, H)


def find_pass_folders(root):
    return sorted(dp for dp, _d, files in os.walk(root) if "VV_dB.tif" in files and "VH_dB.tif" in files)


def make_tiles(env_file=None, folders=None, overlap=None, force=False):
    env = load_env(find_env(env_file))
    vv_rng, vh_rng, T = VV_DB_RANGE, VH_DB_RANGE, TILE_SIZE
    overlap = overlap if overlap is not None else int(env.get("TILE_OVERLAP") or 64)
    if not 0 <= overlap < T:
        raise StopRun("overlap must be between 0 and the tile size")
    if not folders:
        root = os.path.abspath(env.get("S1_DATA_DIR") or DEFAULT_DATA_DIR)
        folders = find_pass_folders(root)
        if not folders:
            raise StopRun(f"No passes with VV_dB.tif + VH_dB.tif under {root}. Run 01_s1_download.ipynb first.")
        print(f"Archive: {root} ({len(folders)} pass(es))")
    print(f"Scaling: VV {vv_rng} dB, VH {vh_rng} dB, {T} px tiles, {overlap} px overlap")

    done, skipped, failed = [], 0, 0
    for folder in folders:
        if os.path.exists(os.path.join(folder, "tiles.csv")) and not force:
            skipped += 1
            continue
        try:
            n, (W, H) = tile_folder(folder, vv_rng, vh_rng, T, overlap)
            print(f"  {os.path.relpath(folder)}: {W} x {H} px -> {n} tiles")
            done.append(folder)
        except Exception as e:
            print(f"  {os.path.relpath(folder)}: FAILED ({e})")
            failed += 1
    print(f"\nTiled {len(done)} pass(es); {skipped} already had tiles (FORCE = True to redo); {failed} failed.")
    return {"folders": list(folders), "tiled": done}

## Run

In [ ]:
try:
    result = make_tiles(ENV_FILE, FOLDERS, OVERLAP, FORCE)
except StopRun as e:
    result = None
    print("⚠️ ", e)

## Look at the result
Tiles from the newest pass, next to each other. They should look like the `xview3_tiles` images:
dark sea, bright ships, same colours.

In [ ]:
from io import BytesIO
from IPython.display import Image as ShowImage, display

if result and result["folders"]:
    folder = sorted(result["folders"], key=lambda f: os.path.basename(f))[-1]
    with open(os.path.join(folder, "tiles.csv"), newline="") as f:
        tiles = list(csv.DictReader(f))
    print(f"{os.path.relpath(folder)}: {len(tiles)} tiles")
    pick = sorted(tiles, key=lambda t: -float(t["valid_fraction"]))[:8]
    thumbs = [Image.open(os.path.join(folder, t["file"])).resize((256, 256)) for t in pick]
    sheet = Image.new("RGB", (4 * 260, ((len(thumbs) + 3) // 4) * 260), (255, 255, 255))
    for i, im in enumerate(thumbs):
        sheet.paste(im, ((i % 4) * 260, (i // 4) * 260))
    buf = BytesIO()
    sheet.save(buf, "PNG")
    display(ShowImage(data=buf.getvalue()))

## From a detection to latitude/longitude
Example: the centre pixel of the first tile. In your own code, do the same with the centre of each
detection box (in tile pixels). Tiles overlap, so merge detections that are within ~50 m of each other.

In [ ]:
if result and result["folders"]:
    t = tiles[0]
    lat, lon = tile_pixel_to_latlon(t, 320, 320)
    print(f"tile {t['tile_id']}, pixel (320, 320) -> {lat:.5f}, {lon:.5f}  (image time {t['acquired_utc']})")